# Week 02 

MSSV: 22302421

Name: Nguyễn Phúc Minh Châu

In [ ]:
%pip install opencv-python

Note: you may need to restart the kernel to use updated packages.


### Lab 1: Capture Frames from Camera

In [ ]:
import cv2

cap = cv2.VideoCapture(0)
ret, frame = cap.read()

if ret:
    cv2.imshow('Frame', frame)
    cv2.waitKey(0)

cap.release()
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /h

### Lab 2: Construct the K Matrix

In [ ]:
import numpy as np

fx, fy = 800, 800 #pixels
cx, cy = 320, 240 #image center

K = np.array([
    [fx, 0, cx],
    [0, fy, cy],
    [0, 0, 1]
], dtype=float)

### Lab 3: Project a 3D Point to 2D

In [ ]:
import numpy as np

X_3d = np.array([1.0, 2.0, 5.0])

#Perspective divide first
x_norm = X_3d[0] / X_3d[2]
y_norm = X_3d[1] / X_3d[2]

p_h = np.array([x_norm, y_norm, 1])
p_2d = K @ p_h

u = int(p_2d[0])
v = int(p_2d[1])

### Lab 4: Load Calibration Data

In [ ]:
import pickle

#Load saved calibration

with open('calib.pkl', 'rb') as f:
    calib_data = pickle.load(f)

K = calib_data['K']
dist = calib_data['dist']

print("K matrix loaded:")
print(K)

### Lab 5: Undistort an Image

In [ ]:
import cv2

#Simple undistortion
undist = cv2.undistort(frame, K, dist)
cv2.imshow('Original', frame)
cv2.imshow('Undistorted', undist)

#Faster for real-time use:
mapx, mapy = cv2.initUndistortRectifyMap(
    K, dist, None, K, 
    (w, h), cv2.CV_32FC1)

undist_fast = cv2.remap(frame, mapx, mapy, 
                        interpolation=cv2.INTER_LINEAR)

### Lab 6: Draw Projected Points on Image

In [ ]:
img = frame.copy()

#u1, v1 = 400, 300
#u2, v2 = 200, 150
projected_point = [(u1, v1), (u2, v2)]

for (u, v) in projected_point:
    cv2.circle(
        img,        #image
        (u, v),     #center pixel
        5,          #radius
        (0, 255, 0),#green color
        -1          #filled circle
    )

cv2.imshow('Projected', img)

### Lab 7: Determine Object Grasp Position

In [ ]:
import cv2
import numpy as np

fx, fy = 500, 500
cx, cy = 320, 240
R_cam_to_robot = np.eye(3) 
t_cam_to_robot = np.array([100, 0, 500])

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break

    # 1. Detect Object
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([0, 120, 70]), np.array([10, 255, 255]))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # 2. Compute Centroid
    for cnt in contours:
        if cv2.contourArea(cnt) > 500:
            M = cv2.moments(cnt)
            if M["m00"] != 0:
                u = int(M["m10"] / M["m00"])
                v = int(M["m01"] / M["m00"])

                # 3. Convert to 3D
                # Giả sử Z là khoảng cách từ camera đến vật (đơn vị: mm hoặc m)
                # Nếu dùng camera RGB-D, Z = depth_frame.get_distance(u, v)
                Z = 1.0 # Ví dụ vật cách camera 1 mét               
                X = (u - cx) * Z / fx
                Y = (v - cy) * Z / fy
                P_camera = np.array([X, Y, Z])
                # 4. OUTPUT ROBOT TARGET (Chuyển sang Robot Base Frame)
                # P_robot = R * P_camera + t
                P_robot = np.dot(R_cam_to_robot, P_camera) + t_cam_to_robot

                # --- HIỂN THỊ KẾT QUẢ ---
                # Vẽ khung và tâm vật thể
                x_b, y_b, w, h = cv2.boundingRect(cnt)
                cv2.rectangle(frame, (x_b, y_b), (x_b + w, y_b + h), (0, 255, 0), 2)
                cv2.circle(frame, (u, v), 5, (0, 0, 255), -1)
                
                # In tọa độ mục tiêu cho Robot
                print(f"Target for Robot: {P_robot}")
                cv2.putText(frame, f"Robot XYZ: {P_robot.astype(int)}", (u-50, v-20), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

    cv2.imshow('3D Projection', frame)
    if cv2.waitKey(1) != -1: break

cap.release()
cv2.destroyAllWindows()

Target for Robot: [1.00056e+02 1.28000e-01 5.01000e+02]
Target for Robot: [1.0005e+02 1.2800e-01 5.0100e+02]
Target for Robot: [1.00048e+02 1.30000e-01 5.01000e+02]
Target for Robot: [1.00044e+02 1.30000e-01 5.01000e+02]
Target for Robot: [1.0005e+02 1.2600e-01 5.0100e+02]
Target for Robot: [1.00038e+02 1.30000e-01 5.01000e+02]
Target for Robot: [1.00024e+02 1.32000e-01 5.01000e+02]
Target for Robot: [ 1.00212e+02 -1.12000e-01  5.01000e+02]
Target for Robot: [1.00044e+02 1.32000e-01 5.01000e+02]
Target for Robot: [ 1.00184e+02 -9.40000e-02  5.01000e+02]
Target for Robot: [1.0003e+02 1.3000e-01 5.0100e+02]
Target for Robot: [ 1.00184e+02 -9.60000e-02  5.01000e+02]
Target for Robot: [1.00028e+02 1.32000e-01 5.01000e+02]
Target for Robot: [ 1.0035e+02 -8.0000e-02  5.0100e+02]
Target for Robot: [ 1.00192e+02 -9.40000e-02  5.01000e+02]
Target for Robot: [1.00026e+02 1.34000e-01 5.01000e+02]
Target for Robot: [ 1.00452e+02 -9.60000e-02  5.01000e+02]
Target for Robot: [ 1.00204e+02 -1.24000e-

### Lab 8: Estimate Extrinsic [R|t] with solvePnP

In [ ]:
import cv2
ret, rvec, tvec = cv2.solvePnP(
    objpoints,   # Nx3 world points
    imgpoints,   # Nx2 image points    
    K,           # camera matrix    
    dist         # distortion coeffs
)

# Convert rotation vector to matrix
R, _ = cv2.Rodrigues(rvec)

### Lab 9: Build a Calibration Dataset

In [1]:
import numpy as np
import cv2
import pickle

# --- CẤU HÌNH ---
CHECKERBOARD = (8, 6) # Số góc trong (ngang-1, dọc-1)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

# Chuẩn bị tọa độ thực tế
objp = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

objpoints = [] 
imgpoints = [] 

cap = cv2.VideoCapture(0) # Mở camera (số 0 là camera mặc định)
print("Hướng dẫn: \n- Nhấn 'S' để lưu frame (cần 20 frame)\n- Nhấn 'Q' để bắt đầu tính toán & thoát")

while len(objpoints) < 20:
    ret, frame = cap.read()
    if not ret: break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    display_frame = frame.copy()
    
    # Tìm góc để hiển thị preview cho người dùng
    found, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)
    if found:
        cv2.drawChessboardCorners(display_frame, CHECKERBOARD, corners, found)

    cv2.putText(display_frame, f"Saved: {len(objpoints)}/20", (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    cv2.imshow('Calibration Lab', display_frame)

    key = cv2.waitKey(1) & 0xFF
    if key == ord('s') and found:
        objpoints.append(objp)
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        imgpoints.append(corners2)
        print(f"Đã lưu frame thứ {len(objpoints)}")
    elif key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# --- TÍNH TOÁN (Chỉ chạy nếu đủ dữ liệu) ---
if len(objpoints) > 0:
    print("Đang tính toán... vui lòng đợi...")
    ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)
    
    print(f"\nRMS Error: {ret}")
    if ret < 1.0:
        with open("calib.pkl", "wb") as f:
            pickle.dump({"K": mtx, "dist": dist}, f)
        print("Thành công! Đã lưu file calib.pkl")
    else:
        print("Cảnh báo: RMS > 1.0, kết quả có thể không chính xác.")
else:
    print("Chưa có dữ liệu để tính toán.")

Hướng dẫn: 
- Nhấn 'S' để lưu frame (cần 20 frame)
- Nhấn 'Q' để bắt đầu tính toán & thoát


QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/zhu/micromamba/envs/tch/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /h

Chưa có dữ liệu để tính toán.


#### Bằng Ảnh

In [2]:
import cv2
import numpy as np
import pickle

# --- CẤU HÌNH ---
CHECKERBOARD = (8, 5)  # Số góc trong của bàn cờ
IMAGE_PATH = 'Calibration.png'
PKL_PATH = 'calib.pkl'

# 1. CHUẨN BỊ DỮ LIỆU GỐC (Object Points)
# Tạo khung tọa độ 3D cho bàn cờ: (0,0,0), (1,0,0), ..., (5,8,0)
objpoints = np.zeros((CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objpoints[:, :2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)

# 2. GIẢ LẬP LƯU FILE .PKL (Dành cho lần chạy đầu tiên)
# Trong thực tế, K và dist đến từ hàm cv2.calibrateCamera()
K_init = np.array([[800, 0, 320], [0, 800, 240], [0, 0, 1]], dtype=np.float32)
dist_init = np.zeros(5, dtype=np.float32)
with open(PKL_PATH, 'wb') as f:
    pickle.dump({'K': K_init, 'dist': dist_init}, f)

# 3. THỰC THI PIPELINE
# Bước A: Load tham số từ file .pkl
with open(PKL_PATH, 'rb') as f:
    calib_data = pickle.load(f)
    K = calib_data['K']
    dist = calib_data['dist']

# Bước B: Phát hiện góc trên ảnh Calibration.png
img = cv2.imread(IMAGE_PATH)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, None)

if ret:
    # Tinh chỉnh tọa độ góc sub-pixel
    imgpoints = cv2.cornerSubPix(
        gray, corners, (11, 11), (-1, -1), 
        (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
    )

    # Bước C: Ước lượng tư thế với solvePnP
    ret, rvec, tvec = cv2.solvePnP(
        objpoints,   # Tọa độ 3D thực tế
        imgpoints,   # Tọa độ 2D trên ảnh
        K,           # Ma trận camera
        dist         # Hệ số biến dạng
    )

    # Bước D: Chuyển đổi Rotation Vector sang Ma trận Quay 3x3
    R, _ = cv2.Rodrigues(rvec)

    # Bước E: Hiển thị kết quả
    print("--- POSE ESTIMATION RESULTS ---")
    print(f"Rotation Matrix (R):\n{R}")
    print(f"\nTranslation Vector (t - [x, y, z]):\n{tvec}")
    
    # (Tùy chọn) Vẽ trục tọa độ lên ảnh để kiểm tra trực quan
    cv2.drawFrameAxes(img, K, dist, rvec, tvec, 3)
    cv2.imshow('Pose Estimation', img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

else:
    print(f"Lỗi: Không thể tìm thấy bàn cờ trong file {IMAGE_PATH}")

--- POSE ESTIMATION RESULTS ---
Rotation Matrix (R):
[[ 1.85364305e-04  9.99999510e-01 -9.72508639e-04]
 [-9.99998935e-01  1.83956116e-04 -1.44788603e-03]
 [-1.44770642e-03  9.72775990e-04  9.99998479e-01]]

Translation Vector (t - [x, y, z]):
[[-4.54161982]
 [ 3.92594479]
 [15.05227032]]
